# Predicting Student Test Scores
## Score: 8.74315

In [1]:
import subprocess
import sys

try:
    import lightgbm as lgb
    from catboost import CatBoostRegressor
    import xgboost as xgb
    from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
    from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression, LogisticRegression
    from sklearn.isotonic import IsotonicRegression
    from sklearn.cluster import KMeans
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lightgbm", "catboost", "xgboost", "--quiet"])
    import lightgbm as lgb
    from catboost import CatBoostRegressor
    import xgboost as xgb
    from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
    from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression, LogisticRegression
    from sklearn.isotonic import IsotonicRegression
    from sklearn.cluster import KMeans

import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

TARGET_MAX = 100.0
TARGET_EPS = 1e-3

def y_to_z(y_arr: np.ndarray) -> np.ndarray:
    y01 = np.clip(y_arr, TARGET_EPS, TARGET_MAX - TARGET_EPS) / TARGET_MAX
    return np.log(y01 / (1.0 - y01))

def z_to_y(z_arr: np.ndarray) -> np.ndarray:
    y01 = 1.0 / (1.0 + np.exp(-z_arr))
    return TARGET_MAX * y01

In [2]:
train = pd.read_csv('playground-series-s6e1/train.csv')
test = pd.read_csv('playground-series-s6e1/test.csv')
test_ids = test['id'].copy()


In [3]:
def create_features(df, train_df=None):
    df = df.copy()
    
    df['internet_access'] = (df['internet_access'] == 'yes').astype(int)
    
    sleep_quality_map = {'poor': 0, 'average': 1, 'good': 2}
    facility_rating_map = {'low': 0, 'medium': 1, 'high': 2}
    exam_difficulty_map = {'easy': 0, 'moderate': 1, 'hard': 2}
    
    df['sleep_quality_ord'] = df['sleep_quality'].map(sleep_quality_map)
    df['facility_rating_ord'] = df['facility_rating'].map(facility_rating_map)
    df['exam_difficulty_ord'] = df['exam_difficulty'].map(exam_difficulty_map)
    
    df['study_efficiency'] = df['study_hours'] * df['class_attendance']
    df['study_sleep_ratio'] = df['study_hours'] / (df['sleep_hours'] + 1e-5)
    df['attendance_facility'] = df['class_attendance'] * df['facility_rating_ord']
    df['sleep_quality_score'] = df['sleep_hours'] * df['sleep_quality_ord']
    df['study_hours_sq'] = df['study_hours'] ** 2
    df['attendance_sq'] = df['class_attendance'] ** 2
    df['study_per_age'] = df['study_hours'] / (df['age'] + 1e-5)
    df['attendance_per_age'] = df['class_attendance'] / (df['age'] + 1e-5)
    df['study_per_attendance'] = df['study_hours'] / (df['class_attendance'] + 1e-5)
    df['sleep_per_age'] = df['sleep_hours'] / (df['age'] + 1e-5)
    
    df['study_hours_cubed'] = df['study_hours'] ** 3
    df['study_attendance_sleep'] = df['study_hours'] * df['class_attendance'] * df['sleep_hours']
    df['efficiency_sleep'] = df['study_efficiency'] * df['sleep_hours']
    df['study_facility'] = df['study_hours'] * df['facility_rating_ord']
    df['difficulty_facility'] = df['exam_difficulty_ord'] * df['facility_rating_ord']
    df['study_attendance_facility'] = df['study_hours'] * df['class_attendance'] * df['facility_rating_ord']
    df['sleep_attendance'] = df['sleep_hours'] * df['class_attendance']
    df['study_difficulty'] = df['study_hours'] * df['exam_difficulty_ord']
    df['attendance_difficulty'] = df['class_attendance'] * df['exam_difficulty_ord']
    df['total_effort'] = df['study_hours'] + df['class_attendance'] / 10
    df['sleep_ratio_sq'] = df['study_sleep_ratio'] ** 2
    df['study_attendance_ratio'] = df['study_hours'] / (df['class_attendance'] + 1e-5)
    df['efficiency_per_sleep'] = df['study_efficiency'] / (df['sleep_hours'] + 1e-5)
    df['study_facility_difficulty'] = df['study_hours'] * df['facility_rating_ord'] * df['exam_difficulty_ord']
    df['attendance_sleep_quality'] = df['class_attendance'] * df['sleep_hours'] * df['sleep_quality_ord']
    
    df['study_sleep_quality'] = df['study_hours'] * df['sleep_quality_ord']
    df['attendance_sleep_quality_ord'] = df['class_attendance'] * df['sleep_quality_ord']
    df['study_facility_sleep_quality'] = df['study_hours'] * df['facility_rating_ord'] * df['sleep_quality_ord']
    df['attendance_difficulty_sleep'] = df['class_attendance'] * df['exam_difficulty_ord'] * df['sleep_hours']
    df['efficiency_difficulty'] = df['study_efficiency'] * df['exam_difficulty_ord']
    df['efficiency_facility'] = df['study_efficiency'] * df['facility_rating_ord']
    df['study_hours_log'] = np.log1p(df['study_hours'])
    df['attendance_log'] = np.log1p(df['class_attendance'])
    df['sleep_hours_log'] = np.log1p(df['sleep_hours'])
    df['age_squared'] = df['age'] ** 2

    df['sleep_opt_dist2_8'] = (df['sleep_hours'] - 8.0) ** 2
    df['sleep_opt_dist2_7'] = (df['sleep_hours'] - 7.0) ** 2
    
    df['study_hours_bin'] = pd.cut(df['study_hours'], bins=5, labels=False, duplicates='drop')
    df['attendance_bin'] = pd.cut(df['class_attendance'], bins=5, labels=False, duplicates='drop')
    df['age_group'] = pd.cut(df['age'], bins=[0, 18, 20, 22, 25], labels=[0, 1, 2, 3], duplicates='drop')
    df['age_group'] = df['age_group'].fillna(2).astype(int)
    
    if train_df is not None:
        top_features_for_interactions = ['study_hours', 'class_attendance', 'sleep_hours', 'study_efficiency', 'study_sleep_ratio']
        for i, feat1 in enumerate(top_features_for_interactions):
            for feat2 in top_features_for_interactions[i+1:]:
                if feat1 in df.columns and feat2 in df.columns:
                    df[f'{feat1}_x_{feat2}'] = df[feat1] * df[feat2]
                    df[f'{feat1}_div_{feat2}'] = df[feat1] / (df[feat2] + 1e-5)
        
        top3_features = ['study_hours', 'class_attendance', 'sleep_hours']
        if all(feat in df.columns for feat in top3_features):
            df['study_attendance_sleep_3way'] = df['study_hours'] * df['class_attendance'] * df['sleep_hours']
        
        if 'study_efficiency' in df.columns and 'sleep_hours' in df.columns and 'facility_rating_ord' in df.columns:
            df['efficiency_sleep_facility_3way'] = df['study_efficiency'] * df['sleep_hours'] * df['facility_rating_ord']
        
        if 'study_hours' in df.columns and 'study_sleep_ratio' in df.columns and 'facility_rating_ord' in df.columns:
            df['study_ratio_facility_3way'] = df['study_hours'] * df['study_sleep_ratio'] * df['facility_rating_ord']
        numeric_cols = ['study_hours', 'class_attendance', 'sleep_hours', 'age']
        cat_cols = ['course', 'study_method', 'gender']
        
        stats_cols = {}
        for cat_col in cat_cols:
            grp = train_df.groupby(cat_col)
            for num_col in numeric_cols:
                stats = grp[num_col].agg(['mean', 'std', 'min', 'max', 'median'])

                mean_map = df[cat_col].map(stats['mean'])
                median_map = df[cat_col].map(stats['median'])

                stats_cols[f'{num_col}_mean_by_{cat_col}'] = mean_map
                stats_cols[f'{num_col}_std_by_{cat_col}'] = df[cat_col].map(stats['std'])
                stats_cols[f'{num_col}_min_by_{cat_col}'] = df[cat_col].map(stats['min'])
                stats_cols[f'{num_col}_max_by_{cat_col}'] = df[cat_col].map(stats['max'])
                stats_cols[f'{num_col}_median_by_{cat_col}'] = median_map
                stats_cols[f'{num_col}_diff_from_mean_{cat_col}'] = df[num_col] - mean_map
                stats_cols[f'{num_col}_diff_from_median_{cat_col}'] = df[num_col] - median_map

                q25 = grp[num_col].quantile(0.25)
                q75 = grp[num_col].quantile(0.75)
                stats_cols[f'{num_col}_q25_by_{cat_col}'] = df[cat_col].map(q25)
                stats_cols[f'{num_col}_q75_by_{cat_col}'] = df[cat_col].map(q75)

        df = pd.concat([df, pd.DataFrame(stats_cols, index=df.index)], axis=1)
        
        cluster_features = ['study_hours', 'class_attendance', 'sleep_hours', 'age']
        kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
        kmeans.fit(train_df[cluster_features])
        df['cluster'] = kmeans.predict(df[cluster_features])
        df['cluster_dist_0'] = np.linalg.norm(df[cluster_features].values - kmeans.cluster_centers_[0], axis=1)
        df['cluster_dist_1'] = np.linalg.norm(df[cluster_features].values - kmeans.cluster_centers_[1], axis=1)
        df['cluster_dist_2'] = np.linalg.norm(df[cluster_features].values - kmeans.cluster_centers_[2], axis=1)
    
    return df

train = create_features(train, train_df=train)
test = create_features(test, train_df=train)

AUGMENT_RATE = 0.0
AUGMENT_NOISE_PCT = 0.01

if AUGMENT_RATE > 0:
    np.random.seed(42)
    numeric_cols_for_aug = ['study_hours', 'class_attendance', 'sleep_hours', 'age']
    augmented_rows = []

    for _ in range(int(len(train) * AUGMENT_RATE)):
        idx = np.random.randint(0, len(train))
        row = train.iloc[idx].copy()

        for col in numeric_cols_for_aug:
            if col in row.index:
                noise = np.random.normal(0, row[col] * AUGMENT_NOISE_PCT)
                row[col] = max(0, row[col] + noise)

        augmented_rows.append(row)

    if augmented_rows:
        train_aug = pd.DataFrame(augmented_rows)
        train = pd.concat([train, train_aug], ignore_index=True)


c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python3

In [4]:
categorical_cols = ['gender', 'course', 'sleep_quality', 'study_method']
target_col = 'exam_score'

iso_kf = KFold(n_splits=5, shuffle=True, random_state=42)

iso_global_oof = np.zeros(len(train), dtype=float)
iso_global_test_accum = np.zeros(len(test), dtype=float)

iso_method_oof = np.zeros(len(train), dtype=float)
iso_method_test_accum = np.zeros(len(test), dtype=float)

x_all = train['study_hours'].to_numpy()
y_all = train[target_col].to_numpy()
m_all = train['study_method'].astype(str).to_numpy()

x_test = test['study_hours'].to_numpy()
m_test = test['study_method'].astype(str).to_numpy()

for tr_idx, va_idx in iso_kf.split(train):
    iso_global = IsotonicRegression(out_of_bounds='clip')
    iso_global.fit(x_all[tr_idx], y_all[tr_idx])

    global_va = iso_global.predict(x_all[va_idx])
    global_test = iso_global.predict(x_test)

    iso_global_oof[va_idx] = global_va
    iso_global_test_accum += global_test / iso_kf.n_splits

    method_va = global_va.copy()
    method_test = global_test.copy()

    for m in np.unique(m_all[tr_idx]):
        tr_m_mask = (m_all[tr_idx] == m)
        if tr_m_mask.sum() < 25:
            continue
        x_m = x_all[tr_idx][tr_m_mask]
        if np.unique(x_m).size < 3:
            continue
        y_m = y_all[tr_idx][tr_m_mask]

        iso_m = IsotonicRegression(out_of_bounds='clip')
        iso_m.fit(x_m, y_m)

        va_m_mask = (m_all[va_idx] == m)
        if va_m_mask.any():
            method_va[va_m_mask] = iso_m.predict(x_all[va_idx][va_m_mask])

        te_m_mask = (m_test == m)
        if te_m_mask.any():
            method_test[te_m_mask] = iso_m.predict(x_test[te_m_mask])

    iso_method_oof[va_idx] = method_va
    iso_method_test_accum += method_test / iso_kf.n_splits

train['iso_study_hours'] = iso_global_oof
test['iso_study_hours'] = iso_global_test_accum

train['iso_study_hours_by_method'] = iso_method_oof
test['iso_study_hours_by_method'] = iso_method_test_accum

for col in categorical_cols:
    train[f'{col}_freq'] = train.groupby(col)[col].transform('count')
    test[f'{col}_freq'] = test[col].map(train.groupby(col)[col].count())

kf_enc = KFold(n_splits=5, shuffle=True, random_state=42)

global_mean = train[target_col].mean()
smoothing = 12.0

for col in categorical_cols:
    train[f'{col}_target'] = 0.0
    test[f'{col}_target'] = 0.0
    train[f'{col}_target_std'] = 0.0
    test[f'{col}_target_std'] = 0.0

    for fold, (train_idx, val_idx) in enumerate(kf_enc.split(train)):
        train_fold = train.iloc[train_idx]
        val_fold = train.iloc[val_idx]

        mean_target = train_fold.groupby(col)[target_col].mean()
        std_target = train_fold.groupby(col)[target_col].std()
        count = train_fold.groupby(col)[target_col].count()

        adaptive_smoothing = smoothing * (1 + 0.5 / (count + 1))
        smoothed = (mean_target * count + global_mean * adaptive_smoothing) / (count + adaptive_smoothing)

        train.loc[val_idx, f'{col}_target'] = val_fold[col].map(smoothed).astype(float)
        train.loc[val_idx, f'{col}_target_std'] = val_fold[col].map(std_target).fillna(0).astype(float)

        test_mean = test[col].map(train_fold.groupby(col)[target_col].mean())
        test_std = test[col].map(train_fold.groupby(col)[target_col].std())
        test_count = test[col].map(train_fold.groupby(col)[target_col].count()).fillna(0)
        test_smoothed = (test_mean * test_count + global_mean * adaptive_smoothing.mean()) / (test_count + adaptive_smoothing.mean())
        test[f'{col}_target'] += test_smoothed.fillna(global_mean).astype(float) / kf_enc.n_splits
        test[f'{col}_target_std'] += test_std.fillna(0).astype(float) / kf_enc.n_splits

categorical_combinations = [
    ('course', 'study_method', 'course_study_method'),
    ('course', 'exam_difficulty', 'course_exam_difficulty'),
    ('course', 'facility_rating', 'course_facility_rating'),
    ('gender', 'course', 'gender_course'),
    ('study_method', 'exam_difficulty', 'study_method_exam_difficulty'),
    ('course', 'sleep_quality', 'course_sleep_quality')
]

for col1, col2, combo_name in categorical_combinations:
    train[combo_name] = train[col1].astype(str) + '_' + train[col2].astype(str)
    test[combo_name] = test[col1].astype(str) + '_' + test[col2].astype(str)
    
    train[f'{combo_name}_target'] = 0.0
    test[f'{combo_name}_target'] = 0.0
    
    for fold, (train_idx, val_idx) in enumerate(kf_enc.split(train)):
        train_fold = train.iloc[train_idx]
        val_fold = train.iloc[val_idx]

        mean_target = train_fold.groupby(combo_name)[target_col].mean()
        count = train_fold.groupby(combo_name)[target_col].count()

        adaptive_smoothing = smoothing * (1 + 0.5 / (count + 1))
        smoothed = (mean_target * count + global_mean * adaptive_smoothing) / (count + adaptive_smoothing)

        train.loc[val_idx, f'{combo_name}_target'] = val_fold[combo_name].map(smoothed).fillna(global_mean).astype(float)

        test_mean = test[combo_name].map(train_fold.groupby(combo_name)[target_col].mean())
        test_count = test[combo_name].map(train_fold.groupby(combo_name)[target_col].count()).fillna(0)
        test_smoothed = (test_mean * test_count + global_mean * adaptive_smoothing.mean()) / (test_count + adaptive_smoothing.mean())
        test[f'{combo_name}_target'] += test_smoothed.fillna(global_mean).astype(float) / kf_enc.n_splits

numeric_cols_to_cap = ['study_hours', 'class_attendance', 'sleep_hours', 'age']
for col in numeric_cols_to_cap:
    q1 = train[col].quantile(0.01)
    q99 = train[col].quantile(0.99)
    train[col] = train[col].clip(lower=q1, upper=q99)
    test[col] = test[col].clip(lower=q1, upper=q99)

combo_cols = [combo[2] for combo in categorical_combinations]
drop_cols = ['id', 'exam_score', 'gender', 'course', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty'] + combo_cols
X = train.drop(drop_cols, axis=1)
y = train[target_col]
X_test = test.drop(['id', 'gender', 'course', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty'], axis=1)

monotone_pos_feats = {'study_hours', 'class_attendance', 'sleep_hours'}

lgb_monotone_constraints = [1 if c in monotone_pos_feats else 0 for c in X.columns]
xgb_monotone_constraints = '(' + ','.join(str(v) for v in lgb_monotone_constraints) + ')'

X_raw = train.drop(['id', 'exam_score'], axis=1)
X_test_raw = test.drop(['id'], axis=1)
cat_feature_indices = [i for i, c in enumerate(X_raw.columns) if X_raw[c].dtype == 'object']

temp_kf = KFold(n_splits=3, shuffle=True, random_state=42)
feature_importance_scores = np.zeros(len(X.columns))
feature_names = X.columns.tolist()

for train_idx, val_idx in temp_kf.split(X):
    X_temp_train, X_temp_val = X.iloc[train_idx], X.iloc[val_idx]
    y_temp_train, y_temp_val = y.iloc[train_idx], y.iloc[val_idx]
    
    temp_model = xgb.XGBRegressor(n_estimators=250, learning_rate=0.1, max_depth=6, 
                                   random_state=42, tree_method='hist', subsample=0.8, colsample_bytree=0.8)
    temp_model.fit(X_temp_train, y_temp_train, 
                   eval_set=[(X_temp_val, y_temp_val)], verbose=False)
    feature_importance_scores += temp_model.feature_importances_ / 3

feature_importance = pd.DataFrame({'feature': feature_names, 'importance': feature_importance_scores})
feature_importance = feature_importance.sort_values('importance', ascending=False)
keep_pct = 0.94
n_keep = int(len(feature_importance) * keep_pct)
selected_features = feature_importance.head(n_keep)['feature'].values
X = X[selected_features]
X_test = X_test[selected_features]

lgb_monotone_constraints = [1 if c in monotone_pos_feats else 0 for c in X.columns]
xgb_monotone_constraints = '(' + ','.join(str(v) for v in lgb_monotone_constraints) + ')'


In [5]:
N_SPLITS = 3
N_BINS = 10

kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
y_bins = pd.qcut(y, q=N_BINS, labels=False, duplicates='drop')


In [6]:
seeds = [42, 123]

xgb_params_base = {
    'n_estimators': 650,
    'learning_rate': 0.02,
    'max_depth': 9,
    'subsample': 0.80,
    'colsample_bytree': 0.80,
    'reg_alpha': 0.10,
    'reg_lambda': 0.10,
    'tree_method': 'hist',
    'device': 'cpu'
}

lgb_params_base = {
    'n_estimators': 700,
    'learning_rate': 0.02,
    'max_depth': 8,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'reg_alpha': 0.08,
    'reg_lambda': 0.08,
    'verbose': -1,
    'n_jobs': -1
}

cat_params_base = {
    'iterations': 650,
    'learning_rate': 0.02,
    'depth': 8,
    'subsample': 0.80,
    'colsample_bylevel': 0.80,
    'l2_leaf_reg': 0.10,
    'loss_function': 'RMSE',
    'verbose': False
}

CATN_ITERS = 800
CATN_EARLY_STOP = 50


In [7]:
all_oof = []
all_test_preds = []
model_names = []

for seed in seeds:
    xgb_params = {**xgb_params_base, 'random_state': seed}
    lgb_params = {**lgb_params_base, 'random_state': seed}
    cat_params = {**cat_params_base, 'random_seed': seed}
    
    xgb_oof = np.zeros(len(train))
    lgb_oof = np.zeros(len(train))
    cat_oof = np.zeros(len(train))
    catn_oof = np.zeros(len(train))
    hgb_oof = np.zeros(len(train))
    
    xgb_test_preds = np.zeros(len(test))
    lgb_test_preds = np.zeros(len(test))
    cat_test_preds = np.zeros(len(test))
    catn_test_preds = np.zeros(len(test))
    hgb_test_preds = np.zeros(len(test))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_bins)):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
        
        dtrain = xgb.DMatrix(X_train_fold, label=y_train_fold.to_numpy())
        dval = xgb.DMatrix(X_val_fold, label=y_val_fold.to_numpy())
        dtest = xgb.DMatrix(X_test)

        xgb_train_params = {
            'objective': 'reg:squarederror',
            'eval_metric': 'rmse',
            'eta': xgb_params['learning_rate'],
            'max_depth': xgb_params['max_depth'],
            'subsample': xgb_params['subsample'],
            'colsample_bytree': xgb_params['colsample_bytree'],
            'reg_alpha': xgb_params['reg_alpha'],
            'reg_lambda': xgb_params['reg_lambda'],
            'tree_method': xgb_params.get('tree_method', 'hist'),
            'device': xgb_params.get('device', 'cpu'),
            'seed': seed,
            'verbosity': 0
        }

        booster = xgb.train(
            params=xgb_train_params,
            dtrain=dtrain,
            num_boost_round=int(xgb_params['n_estimators']),
            evals=[(dval, 'val')],
            early_stopping_rounds=50,
            verbose_eval=False
        )

        best_it = getattr(booster, 'best_iteration', None)
        if best_it is None:
            val_pred = booster.predict(dval)
            test_pred = booster.predict(dtest)
        else:
            val_pred = booster.predict(dval, iteration_range=(0, best_it + 1))
            test_pred = booster.predict(dtest, iteration_range=(0, best_it + 1))

        xgb_oof[val_idx] = val_pred
        xgb_test_preds += test_pred / kf.n_splits

        lgb_model = lgb.LGBMRegressor(**lgb_params, monotone_constraints=lgb_monotone_constraints)
        lgb_model.fit(X_train_fold, y_train_fold,
                      eval_set=[(X_val_fold, y_val_fold)],
                      callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
        lgb_oof[val_idx] = lgb_model.predict(X_val_fold)
        lgb_test_preds += lgb_model.predict(X_test) / kf.n_splits
        
        cat_model = CatBoostRegressor(**cat_params)
        cat_model.fit(X_train_fold, y_train_fold,
                      eval_set=(X_val_fold, y_val_fold),
                      early_stopping_rounds=50)
        cat_oof[val_idx] = cat_model.predict(X_val_fold)
        cat_test_preds += cat_model.predict(X_test) / kf.n_splits

        catn_model = CatBoostRegressor(
            iterations=CATN_ITERS,
            learning_rate=0.03,
            depth=8,
            loss_function='RMSE',
            random_seed=seed,
            subsample=0.85,
            colsample_bylevel=0.85,
            l2_leaf_reg=3.0,
            random_strength=0.2,
            verbose=False
        )
        catn_model.fit(
            X_raw.iloc[train_idx],
            y_train_fold,
            eval_set=(X_raw.iloc[val_idx], y_val_fold),
            cat_features=cat_feature_indices,
            early_stopping_rounds=CATN_EARLY_STOP
        )
        catn_oof[val_idx] = catn_model.predict(X_raw.iloc[val_idx])
        catn_test_preds += catn_model.predict(X_test_raw) / kf.n_splits
        
        hgb_model = HistGradientBoostingRegressor(
            max_iter=250, learning_rate=0.03, max_depth=8,
            random_state=seed, l2_regularization=0.1
        )
        hgb_model.fit(X_train_fold, y_train_fold)
        hgb_oof[val_idx] = hgb_model.predict(X_val_fold)
        hgb_test_preds += hgb_model.predict(X_test) / kf.n_splits
    
    all_oof.append(xgb_oof)
    all_oof.append(lgb_oof)
    all_oof.append(cat_oof)
    all_oof.append(catn_oof)
    all_oof.append(hgb_oof)
    all_test_preds.append(xgb_test_preds)
    all_test_preds.append(lgb_test_preds)
    all_test_preds.append(cat_test_preds)
    all_test_preds.append(catn_test_preds)
    all_test_preds.append(hgb_test_preds)
    model_names.extend([f'xgb_{seed}', f'lgb_{seed}', f'cat_{seed}', f'catn_{seed}', f'hgb_{seed}'])
    
    xgb_rmse = np.sqrt(mean_squared_error(y, xgb_oof))
    lgb_rmse = np.sqrt(mean_squared_error(y, lgb_oof))
    cat_rmse = np.sqrt(mean_squared_error(y, cat_oof))
    catn_rmse = np.sqrt(mean_squared_error(y, catn_oof))
    hgb_rmse = np.sqrt(mean_squared_error(y, hgb_oof))
    print(f'Seed {seed} - XGB: {xgb_rmse:.5f}, LGB: {lgb_rmse:.5f}, Cat: {cat_rmse:.5f}, CatN: {catn_rmse:.5f}, HGB: {hgb_rmse:.5f}')

all_oof = np.column_stack(all_oof)
all_test_preds = np.column_stack(all_test_preds)


Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[700]	valid_0's l2: 77.7178
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[700]	valid_0's l2: 77.8737
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[700]	valid_0's l2: 77.634
Seed 42 - XGB: 8.78540, LGB: 8.81713, Cat: 8.83037, CatN: 8.80512, HGB: 8.84121
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[700]	valid_0's l2: 77.6871
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[699]	valid_0's l2: 77.8926
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[700]	valid_0's l2: 77.6267
Seed 123 - XGB: 8.78518, LGB: 8.81677, Cat: 8.82974, CatN: 8.80541, HGB: 8.84180


In [8]:
stacking_oof = np.zeros(len(train))
stacking_test_preds = np.zeros(len(test))

stack_cols = model_names
all_test_preds_df = pd.DataFrame(all_test_preds, columns=stack_cols)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_bins)):
    X_stack_train = all_oof[train_idx]
    X_stack_val = all_oof[val_idx]
    y_stack_train = y.iloc[train_idx]
    y_stack_val = y.iloc[val_idx]

    X_stack_train_df = pd.DataFrame(X_stack_train, columns=stack_cols)
    X_stack_val_df = pd.DataFrame(X_stack_val, columns=stack_cols)

    scaler = StandardScaler()
    X_stack_train_scaled = scaler.fit_transform(X_stack_train)
    X_stack_val_scaled = scaler.transform(X_stack_val)

    ridge = Ridge(alpha=2.5, random_state=42)
    lasso = Lasso(alpha=0.25, random_state=42, max_iter=3000)
    elastic = ElasticNet(alpha=0.4, l1_ratio=0.6, random_state=42, max_iter=3000)

    lgb_meta = lgb.LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        random_state=42,
        verbose=-1,
        subsample=0.85,
        colsample_bytree=0.85
    )
    cat_meta = CatBoostRegressor(
        iterations=300,
        learning_rate=0.05,
        depth=5,
        random_seed=42,
        subsample=0.85,
        colsample_bylevel=0.85,
        verbose=False
    )
    
    ridge.fit(X_stack_train_scaled, y_stack_train)
    lasso.fit(X_stack_train_scaled, y_stack_train)
    elastic.fit(X_stack_train_scaled, y_stack_train)

    dtrain_meta = xgb.DMatrix(X_stack_train_df, label=y_stack_train.to_numpy())
    dval_meta = xgb.DMatrix(X_stack_val_df, label=y_stack_val.to_numpy())
    dtest_meta = xgb.DMatrix(all_test_preds_df)

    xgb_meta_params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'eta': 0.05,
        'max_depth': 5,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'tree_method': 'hist',
        'seed': 42,
        'verbosity': 0
    }

    booster_meta = xgb.train(
        params=xgb_meta_params,
        dtrain=dtrain_meta,
        num_boost_round=300,
        evals=[(dval_meta, 'val')],
        early_stopping_rounds=50,
        verbose_eval=False
    )

    best_it_meta = getattr(booster_meta, 'best_iteration', None)
    if best_it_meta is None:
        xgb_meta_val = booster_meta.predict(dval_meta)
        xgb_meta_test = booster_meta.predict(dtest_meta)
    else:
        xgb_meta_val = booster_meta.predict(dval_meta, iteration_range=(0, best_it_meta + 1))
        xgb_meta_test = booster_meta.predict(dtest_meta, iteration_range=(0, best_it_meta + 1))

    lgb_meta.fit(
        X_stack_train_df,
        y_stack_train,
        eval_set=[(X_stack_val_df, y_stack_val)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
    )
    cat_meta.fit(
        X_stack_train_df,
        y_stack_train,
        eval_set=(X_stack_val_df, y_stack_val),
        early_stopping_rounds=50
    )

    ridge_pred = ridge.predict(X_stack_val_scaled)
    lasso_pred = lasso.predict(X_stack_val_scaled)
    elastic_pred = elastic.predict(X_stack_val_scaled)
    xgb_meta_pred = xgb_meta_val
    lgb_meta_pred = lgb_meta.predict(X_stack_val_df)
    cat_meta_pred = cat_meta.predict(X_stack_val_df)
    
    meta_preds = np.column_stack([ridge_pred, lasso_pred, elastic_pred, xgb_meta_pred, lgb_meta_pred, cat_meta_pred])
    meta_rmses = [np.sqrt(mean_squared_error(y.iloc[val_idx], pred)) for pred in meta_preds.T]
    meta_weights = np.array([1/rmse for rmse in meta_rmses])
    meta_weights = meta_weights / meta_weights.sum()
    
    stacking_oof[val_idx] = np.average(meta_preds, axis=1, weights=meta_weights)
    
    all_test_preds_scaled = scaler.transform(all_test_preds)
    test_meta_preds = np.column_stack([
        ridge.predict(all_test_preds_scaled),
        lasso.predict(all_test_preds_scaled),
        elastic.predict(all_test_preds_scaled),
        xgb_meta_test,
        lgb_meta.predict(all_test_preds_df),
        cat_meta.predict(all_test_preds_df)
    ])
    stacking_test_preds += np.average(test_meta_preds, axis=1, weights=meta_weights) / kf.n_splits

stacking_rmse = np.sqrt(mean_squared_error(y, stacking_oof))
print(f'Stacking OOF RMSE: {stacking_rmse:.5f}')


Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[292]	valid_0's l2: 77.0442
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[189]	valid_0's l2: 77.2782
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[288]	valid_0's l2: 77.0047
Stacking OOF RMSE: 8.77968


In [9]:
X_blend_oof = np.column_stack([all_oof, stacking_oof])
X_blend_test = np.column_stack([all_test_preds, stacking_test_preds])

blender = LinearRegression(positive=True)
blender.fit(X_blend_oof, y.to_numpy())

blend_oof = blender.predict(X_blend_oof)
blend_test = blender.predict(X_blend_test)

print(f'Blender OOF RMSE (pre-calibration): {np.sqrt(mean_squared_error(y, blend_oof)):.5f}')

cal_oof = np.zeros_like(blend_oof)
cal_test_accum = np.zeros_like(blend_test)

y_np = y.to_numpy()

for tr_idx, va_idx in kf.split(X, y_bins):
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(blend_oof[tr_idx], y_np[tr_idx])
    cal_oof[va_idx] = iso.predict(blend_oof[va_idx])
    cal_test_accum += iso.predict(blend_test) / kf.n_splits

cal_oof = np.clip(cal_oof, 0, 100)
cal_test = np.clip(cal_test_accum, 0, 100)

final_rmse = np.sqrt(mean_squared_error(y_np, cal_oof))
print(f'Fold-calibrated Final OOF RMSE: {final_rmse:.5f}')

y_is_100 = (y_np >= 99.999).astype(int)

p100_oof = np.zeros_like(cal_oof)
p100_test_accum = np.zeros_like(cal_test)

for tr_idx, va_idx in kf.split(X, y_bins):
    clf = LogisticRegression(
        C=1.0,
        solver='lbfgs',
        max_iter=200,
        class_weight='balanced',
        random_state=42
    )
    clf.fit(cal_oof[tr_idx].reshape(-1, 1), y_is_100[tr_idx])
    p100_oof[va_idx] = clf.predict_proba(cal_oof[va_idx].reshape(-1, 1))[:, 1]
    p100_test_accum += clf.predict_proba(cal_test.reshape(-1, 1))[:, 1] / kf.n_splits

CEILING_STRENGTH = 0.25
P100_THRESHOLD = 0.80
NEAR_CEILING = 97.0

p100_test = np.clip(p100_test_accum, 0, 1)

gate_oof = ((p100_oof >= P100_THRESHOLD) & (cal_oof >= NEAR_CEILING)).astype(float)
gate_test = ((p100_test >= P100_THRESHOLD) & (cal_test >= NEAR_CEILING)).astype(float)

final_oof = cal_oof + CEILING_STRENGTH * gate_oof * p100_oof * (100.0 - cal_oof)
final_predictions = cal_test + CEILING_STRENGTH * gate_test * p100_test * (100.0 - cal_test)

final_oof = np.clip(final_oof, 0, 100)
final_predictions = np.clip(final_predictions, 0, 100)

final_rmse_ceiling = np.sqrt(mean_squared_error(y_np, final_oof))
print(f'Final OOF RMSE after ceiling head: {final_rmse_ceiling:.5f}')

submission = pd.DataFrame({
    'id': test_ids,
    'exam_score': final_predictions
})
submission.to_csv('submission.csv', index=False)


Blender OOF RMSE (pre-calibration): 8.77788
Fold-calibrated Final OOF RMSE: 8.78101
Final OOF RMSE after ceiling head: 8.78110


In [10]:
print('Final RMSE already printed in Cell 9 after fold-wise calibration.')


Final RMSE already printed in Cell 9 after fold-wise calibration.


In [11]:
import winsound
import time
notes = [(523, 200), (659, 200), (784, 200), (1047, 400), (784, 200), (1047, 600)]
for freq, dur in notes:
    winsound.Beep(freq, dur)
    time.sleep(0.05)
